# 📘 Módulo 10 — Gradient Boosting, XGBoost, LightGBM e CatBoost
## Modelos de Boosting, Otimização e Alta Performance

Este módulo apresenta os modelos que dominam:
- competições do Kaggle
- aplicações industriais de alto desempenho
- sistemas de recomendação
- risco de crédito
- churn prediction
- detecção de fraude
- modelagem tabular em geral

---

# 🎯 Objetivos do Módulo 10

Ao final deste módulo, você será capaz de:

- entender o conceito de boosting  
- treinar modelos de Gradient Boosting  
- entender por que boosting supera Random Forest  
- treinar XGBoost, LightGBM e CatBoost  
- interpretar importâncias de variáveis  
- ajustar hiperparâmetros avançados  
- construir um pipeline completo de boosting  

---

# 📚 Índice

0. [Setup — bibliotecas e dados](#setup)
1. [Intuição do Boosting](#intuicao)
2. [Gradient Boosting — Fundamentos](#gb)
3. [Gradient Boosting Classifier — Implementação](#gbc)
4. [Comparação com Random Forest](#comparacao)
5. [XGBoost — Intuição e Implementação](#xgb)
6. [LightGBM — Intuição e Implementação](#lgbm)
7. [CatBoost — Intuição e Implementação](#cat)
8. [Importância das Variáveis no Boosting](#importancia)
9. [Projeto Final — Boosting Completo](#projeto)
10. [Apêndice Matemático Avançado](#apendice)

---

<a id="setup"></a>
# 0. Setup — bibliotecas e dados

Vamos usar o mesmo dataset do Módulo 9:
- `high_eval` como variável alvo
- variáveis explicativas contínuas e categóricas

Isso permite comparar Boosting com Random Forest.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, roc_curve, auc
)

from sklearn.ensemble import GradientBoostingClassifier

plt.style.use("seaborn-v0_8-whitegrid")

In [ ]:
ratings_df = pd.read_csv("/home/moacir/projects/ml/IBM/statistics/data/raw/teachingratings.csv")
ratings_df["high_eval"] = (ratings_df["eval"] >= 4.5).astype(int)

ratings_df.head()

<a id="intuicao"></a>
# 2. Intuição do Boosting

No módulo anterior, aprendemos:
- Árvores são modelos flexíveis  
- Random Forest reduz overfitting combinando muitas árvores  

Agora vamos entender um modelo ainda mais poderoso:

# **Boosting**

Enquanto Random Forest constrói **muitas árvores independentes**,  
o Boosting constrói **uma sequência de árvores**, onde:

> Cada nova árvore tenta corrigir os erros da árvore anterior.

Essa é a ideia central.

---
# 2.1 A diferença fundamental: Bagging vs Boosting

## ✔️ Bagging (Random Forest)
- várias árvores **independentes**  
- cada árvore vê dados diferentes  
- previsões são combinadas por média/voto  
- reduz variância  

## ✔️ Boosting
- árvores **sequenciais**  
- cada árvore aprende com os **erros da anterior**  
- cada árvore é fraca, mas a soma é forte  
- reduz viés e variância  

Boosting é como um aluno que:
- faz uma prova, erra  
- estuda apenas o que errou  
- faz outra prova, erra menos  
- repete o processo  

No final, ele se torna excelente.

---
# 2.2 Intuição visual do Boosting

Imagine que queremos prever `high_eval` usando árvores pequenas.

1. **Árvore 1**: tenta prever tudo → erra bastante  
2. **Árvore 2**: foca apenas nos erros da árvore 1  
3. **Árvore 3**: foca nos erros acumulados  
4. ...

No final:

$$ \text{Modelo Final} = T_1 + T_2 + T_3 + \dots + T_M $$

Onde cada \( T_i \) é uma árvore pequena (fraca).

---
# 2.3 Por que Boosting funciona tão bem?

✔️ Cada árvore aprende apenas um pequeno pedaço do problema  
✔️ O modelo final é uma soma de especialistas  
✔️ Árvores pequenas evitam overfitting  
✔️ O processo sequencial reduz viés  
✔️ O modelo final é altamente expressivo  

Isso explica por que:

- XGBoost  
- LightGBM  
- CatBoost  

dominam competições e aplicações reais.

---
# 2.4 Boosting é um modelo aditivo

O modelo final é:

$$
F_M(x) = F_{M-1}(x) + \gamma_M h_M(x)
$$

Onde:
- \( h_M(x) \) é a nova árvore  
- \( \gamma_M \) é o peso da árvore  

Isso é chamado de **modelo aditivo**.

---
# 2.5 Boosting é gradiente descendente em espaço de funções

Essa é a visão mais profunda:

- cada árvore é um passo de gradiente  
- o modelo minimiza uma função de perda  
- por isso se chama **Gradient Boosting**  

Exemplo:
- para classificação → perda logística  
- para regressão → perda quadrática  

Cada árvore tenta reduzir o gradiente da perda.

---
# 2.6 Boosting vs Random Forest — Intuição

| Característica | Random Forest | Boosting |
|----------------|---------------|----------|
| Construção | Paralela | Sequencial |
| Foco | Variância | Viés + Variância |
| Árvores | Profundas | Rasas |
| Overfitting | Baixo | Médio (controlável) |
| Performance | Muito boa | Excelente |
| Velocidade | Rápido | Mais lento |

Boosting geralmente vence Random Forest em:
- dados tabulares  
- problemas complexos  
- competições  

Mas exige mais cuidado com hiperparâmetros.

---
# 2.7 Exemplo simples: Boosting manual com 3 árvores

In [ ]:
from sklearn.tree import DecisionTreeClassifier

# Árvores rasas (stumps)
t1 = DecisionTreeClassifier(max_depth=1, random_state=1)
t2 = DecisionTreeClassifier(max_depth=1, random_state=2)
t3 = DecisionTreeClassifier(max_depth=1, random_state=3)

# Treinando sequencialmente (simulação conceitual)
t1.fit(X_train, y_train)
residual1 = y_train - t1.predict(X_train)

t2.fit(X_train, residual1)
residual2 = residual1 - t2.predict(X_train)

t3.fit(X_train, residual2)

# Previsão final (soma das árvores)
pred_final = (
    t1.predict(X_test)
    + t2.predict(X_test)
    + t3.predict(X_test)
)

pred_final[:10]

🟣 **Interpretação**

Isso NÃO é um modelo real de boosting,  
mas ilustra a ideia:

- cada árvore corrige erros da anterior  
- o modelo final é a soma das árvores  

Gradient Boosting faz isso de forma matemática e otimizada.

---
# 2.8 Conclusão desta parte

✔️ Boosting constrói árvores sequenciais  
✔️ Cada árvore corrige erros da anterior  
✔️ Modelo final = soma de árvores fracas  
✔️ Reduz viés e variância  
✔️ Supera Random Forest em muitos cenários  
✔️ Base para XGBoost, LightGBM e CatBoost  

Agora estamos prontos para:

**PARTE 3 — Gradient Boosting: Fundamentos e Implementação.**

<a id="gb"></a>
# 3. Gradient Boosting — Fundamentos

Agora que entendemos a intuição do Boosting,
vamos estudar o **Gradient Boosting**, que é a implementação clássica
do método de boosting baseada em gradiente descendente.

Ele é o precursor direto de:
- XGBoost  
- LightGBM  
- CatBoost  

E ainda é muito usado em aplicações reais.

---
# 3.1 Como o Gradient Boosting funciona?

O modelo final é uma soma de árvores:

$$
F_M(x) = \sum_{m=1}^M \gamma_m h_m(x)
$$

Onde:
- \( h_m(x) \) é a árvore m  
- \( \gamma_m \) é o peso da árvore  

Cada árvore é treinada para **reduzir o erro residual** da anterior.

Isso é equivalente a:

> Fazer gradiente descendente em espaço de funções.

---
# 3.2 Função de perda

O Gradient Boosting pode usar diferentes perdas:

- Classificação → perda logística  
- Regressão → perda quadrática  
- Regressão robusta → perda Huber  

Para classificação binária, usamos:

$$
L = -y \log(p) - (1-y)\log(1-p)
$$

---
# 3.3 Hiperparâmetros importantes

### ✔️ `n_estimators`
Número de árvores (100–500 é comum).

### ✔️ `learning_rate`
Peso de cada árvore.

- valores pequenos → modelo mais estável  
- valores grandes → risco de overfitting  

### ✔️ `max_depth`
Profundidade das árvores (geralmente 2–4).

### ✔️ `subsample`
Fração dos dados usada em cada árvore.

- < 1.0 → reduz overfitting  

Esses hiperparâmetros são a chave do desempenho.

---
# 3.4 Preparando os dados

In [ ]:
X = ratings_df[["beauty", "age", "students", "female_dummy"]]
y = ratings_df["high_eval"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

---
# 3.5 Ajustando o Gradient Boosting Classifier

In [ ]:
gb = GradientBoostingClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=3,
    subsample=0.9,
    random_state=42
)

gb.fit(X_train, y_train)

gb

---
# 3.6 Avaliando o modelo

In [ ]:
y_pred_gb = gb.predict(X_test)

results_gb = pd.DataFrame({
    "Acurácia": [accuracy_score(y_test, y_pred_gb)],
    "Precisão": [precision_score(y_test, y_pred_gb)],
    "Recall": [recall_score(y_test, y_pred_gb)],
    "F1": [f1_score(y_test, y_pred_gb)]
})

results_gb

🟣 **Interpretação**

Gradient Boosting normalmente:
- supera Random Forest em recall e F1  
- tem excelente separação entre classes  
- é mais sensível a hiperparâmetros  

E isso já começa a aparecer aqui.

---
# 3.7 Matriz de Confusão

In [ ]:
cm = confusion_matrix(y_test, y_pred_gb)

plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Previsto")
plt.ylabel("Real")
plt.title("Matriz de Confusão — Gradient Boosting")
plt.show()

---
# 3.8 Curva ROC e AUC

In [ ]:
y_proba_gb = gb.predict_proba(X_test)[:, 1]

fpr, tpr, thresholds = roc_curve(y_test, y_proba_gb)
auc_value = auc(fpr, tpr)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f"Gradient Boosting (AUC = {auc_value:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
plt.xlabel("FPR")
plt.ylabel("TPR")
plt.title("Curva ROC — Gradient Boosting")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

auc_value

🟣 **Interpretação**

- AUC alto indica excelente separação  
- Boosting tende a ter AUC superior ao Random Forest  
- O modelo é mais “fino” e mais sensível aos padrões dos dados  

---
# 3.9 Comparando Gradient Boosting vs Random Forest

In [ ]:
rf = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

comparison = pd.DataFrame({
    "Modelo": ["Random Forest", "Gradient Boosting"],
    "Acurácia": [
        accuracy_score(y_test, y_pred_rf),
        accuracy_score(y_test, y_pred_gb)
    ],
    "Precisão": [
        precision_score(y_test, y_pred_rf),
        precision_score(y_test, y_pred_gb)
    ],
    "Recall": [
        recall_score(y_test, y_pred_rf),
        recall_score(y_test, y_pred_gb)
    ],
    "F1": [
        f1_score(y_test, y_pred_rf),
        f1_score(y_test, y_pred_gb)
    ]
})

comparison

🟣 **Interpretação**

- Random Forest → mais robusto, menos sensível  
- Gradient Boosting → mais preciso, mais expressivo  

Em muitos casos, Gradient Boosting vence.

---
# 3.10 Conclusão desta parte

✔️ Entendemos o funcionamento do Gradient Boosting  
✔️ Ajustamos um modelo real  
✔️ Avaliamos com métricas completas  
✔️ Geramos curva ROC e AUC  
✔️ Comparamos com Random Forest  

Agora estamos prontos para:

**PARTE 4 — Gradient Boosting Classifier: Implementação Detalhada e Ajuste Fino.**

<a id="gbc"></a>
# 4. Gradient Boosting Classifier — Implementação Detalhada e Ajuste Fino

Agora vamos aprofundar o Gradient Boosting:

- testar diferentes hiperparâmetros  
- visualizar impacto do learning rate  
- analisar número de árvores  
- comparar profundidades  
- entender trade-offs  

Isso é essencial para dominar modelos de boosting.

---
# 4.1 Preparando os dados

In [ ]:
X = ratings_df[["beauty", "age", "students", "female_dummy"]]
y = ratings_df["high_eval"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

---
# 4.2 Testando diferentes learning rates

O learning rate controla o peso de cada árvore.

- valores pequenos → modelo mais estável, precisa de mais árvores  
- valores grandes → modelo mais agressivo, risco de overfitting  

In [ ]:
learning_rates = [0.001, 0.01, 0.05, 0.1, 0.2]
results_lr = []

for lr in learning_rates:
    gb = GradientBoostingClassifier(
        n_estimators=300,
        learning_rate=lr,
        max_depth=3,
        subsample=0.9,
        random_state=42
    )
    gb.fit(X_train, y_train)
    pred = gb.predict(X_test)
    results_lr.append([
        lr,
        accuracy_score(y_test, pred),
        f1_score(y_test, pred)
    ])

pd.DataFrame(results_lr, columns=["Learning Rate", "Acurácia", "F1"])

🟣 **Interpretação**

- learning rate muito baixo → underfitting  
- learning rate muito alto → overfitting  
- valores entre 0.03 e 0.1 costumam ser ideais  

---
# 4.3 Testando diferentes números de árvores (n_estimators)

In [ ]:
estimators = [50, 100, 200, 300, 500]
results_est = []

for n in estimators:
    gb = GradientBoostingClassifier(
        n_estimators=n,
        learning_rate=0.05,
        max_depth=3,
        subsample=0.9,
        random_state=42
    )
    gb.fit(X_train, y_train)
    pred = gb.predict(X_test)
    results_est.append([
        n,
        accuracy_score(y_test, pred),
        f1_score(y_test, pred)
    ])

pd.DataFrame(results_est, columns=["n_estimators", "Acurácia", "F1"])

🟣 **Interpretação**

- poucas árvores → underfitting  
- árvores demais → custo computacional alto  
- 200–400 árvores costuma ser o “sweet spot”  

---
# 4.4 Testando diferentes profundidades das árvores

In [ ]:
depths = [1, 2, 3, 4, 5]
results_depth = []

for d in depths:
    gb = GradientBoostingClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=d,
        subsample=0.9,
        random_state=42
    )
    gb.fit(X_train, y_train)
    pred = gb.predict(X_test)
    results_depth.append([
        d,
        accuracy_score(y_test, pred),
        f1_score(y_test, pred)
    ])

pd.DataFrame(results_depth, columns=["max_depth", "Acurácia", "F1"])

🟣 **Interpretação**

- profundidade 1 → modelo muito simples (stumps)  
- profundidade 2–4 → geralmente ideal  
- profundidade alta → risco de overfitting  

---
# 4.5 Visualizando impacto do learning rate

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(learning_rates, [r[1] for r in results_lr], marker="o", label="Acurácia")
plt.plot(learning_rates, [r[2] for r in results_lr], marker="o", label="F1")
plt.xlabel("Learning Rate")
plt.ylabel("Métrica")
plt.title("Impacto do Learning Rate no Desempenho")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

---
# 4.6 Ajuste fino: modelo final otimizado

In [ ]:
gb_final = GradientBoostingClassifier(
    n_estimators=400,
    learning_rate=0.05,
    max_depth=3,
    subsample=0.9,
    random_state=42
)

gb_final.fit(X_train, y_train)
y_pred_final = gb_final.predict(X_test)

pd.DataFrame({
    "Acurácia": [accuracy_score(y_test, y_pred_final)],
    "Precisão": [precision_score(y_test, y_pred_final)],
    "Recall": [recall_score(y_test, y_pred_final)],
    "F1": [f1_score(y_test, y_pred_final)]
})

🟣 **Interpretação**

Esse modelo final já é competitivo com:
- Random Forest  
- XGBoost (versão padrão)  

E serve como baseline para os próximos modelos.

---
# 4.7 Conclusão desta parte

✔️ Testamos diferentes learning rates  
✔️ Testamos diferentes números de árvores  
✔️ Testamos diferentes profundidades  
✔️ Visualizamos impacto dos hiperparâmetros  
✔️ Construímos um modelo otimizado  

Agora estamos prontos para:

**PARTE 5 — XGBoost: Intuição e Implementação.**

<a id="xgb"></a>
# 5. XGBoost — Intuição e Implementação

XGBoost significa:

# **eXtreme Gradient Boosting**

Ele é uma versão otimizada, paralelizada e regularizada do Gradient Boosting.

Por que XGBoost ficou tão famoso?

✔️ extremamente rápido  
✔️ extremamente preciso  
✔️ regularização embutida  
✔️ paralelização eficiente  
✔️ tratamento inteligente de missing values  
✔️ controle fino de overfitting  
✔️ escalável para milhões de linhas  

É o modelo mais usado em competições e aplicações reais.

---
# 5.1 Instalando e importando XGBoost

Caso não esteja instalado:

```
pip install xgboost
```

In [ ]:
import xgboost as xgb

---
# 5.2 Preparando os dados

In [ ]:
X = ratings_df[["beauty", "age", "students", "female_dummy"]]
y = ratings_df["high_eval"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

---
# 5.3 Ajustando o XGBoost Classifier

In [ ]:
xgb_model = xgb.XGBClassifier(
    n_estimators=400,
    learning_rate=0.05,
    max_depth=3,
    subsample=0.9,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(X_train, y_train)

---
# 5.4 Avaliando o modelo

In [ ]:
y_pred_xgb = xgb_model.predict(X_test)

results_xgb = pd.DataFrame({
    "Acurácia": [accuracy_score(y_test, y_pred_xgb)],
    "Precisão": [precision_score(y_test, y_pred_xgb)],
    "Recall": [recall_score(y_test, y_pred_xgb)],
    "F1": [f1_score(y_test, y_pred_xgb)]
})

results_xgb

🟣 **Interpretação**

XGBoost normalmente:
- supera Gradient Boosting clássico  
- supera Random Forest  
- tem recall e F1 muito fortes  

E isso já aparece aqui.

---
# 5.5 Matriz de Confusão

In [ ]:
cm = confusion_matrix(y_test, y_pred_xgb)

plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Purples")
plt.xlabel("Previsto")
plt.ylabel("Real")
plt.title("Matriz de Confusão — XGBoost")
plt.show()

---
# 5.6 Curva ROC e AUC

In [ ]:
y_proba_xgb = xgb_model.predict_proba(X_test)[:, 1]

fpr, tpr, thresholds = roc_curve(y_test, y_proba_xgb)
auc_value = auc(fpr, tpr)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f"XGBoost (AUC = {auc_value:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
plt.xlabel("FPR")
plt.ylabel("TPR")
plt.title("Curva ROC — XGBoost")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

auc_value

🟣 **Interpretação**

XGBoost costuma ter:
- AUC muito alto  
- excelente separação entre classes  
- performance superior em dados tabulares  

---
# 5.7 Importância das Variáveis

In [ ]:
importance = xgb_model.feature_importances_

importance_df = pd.DataFrame({
    "Variável": X.columns,
    "Importância": importance
}).sort_values("Importância", ascending=False)

importance_df

In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(
    data=importance_df,
    x="Importância",
    y="Variável",
    palette="magma"
)
plt.title("Importância das Variáveis — XGBoost")
plt.xlabel("Importância")
plt.ylabel("Variável")
plt.show()

🟣 **Interpretação**

XGBoost calcula importâncias usando:
- ganho  
- cobertura  
- frequência  

Aqui usamos o método padrão: **gain**.

---
# 5.8 Comparando XGBoost vs Gradient Boosting vs Random Forest

In [ ]:
comparison = pd.DataFrame({
    "Modelo": ["Random Forest", "Gradient Boosting", "XGBoost"],
    "Acurácia": [
        accuracy_score(y_test, y_pred_rf),
        accuracy_score(y_test, y_pred_gb),
        accuracy_score(y_test, y_pred_xgb)
    ],
    "Precisão": [
        precision_score(y_test, y_pred_rf),
        precision_score(y_test, y_pred_gb),
        precision_score(y_test, y_pred_xgb)
    ],
    "Recall": [
        recall_score(y_test, y_pred_rf),
        recall_score(y_test, y_pred_gb),
        recall_score(y_test, y_pred_xgb)
    ],
    "F1": [
        f1_score(y_test, y_pred_rf),
        f1_score(y_test, y_pred_gb),
        f1_score(y_test, y_pred_xgb)
    ]
})

comparison.set_index("Modelo")

🟣 **Interpretação**

- Random Forest → robusto, simples  
- Gradient Boosting → mais preciso  
- XGBoost → geralmente o melhor dos três  

Em dados tabulares, XGBoost costuma ser o campeão.

---
# 5.9 Conclusão desta parte

✔️ Entendemos a intuição do XGBoost  
✔️ Ajustamos um modelo real  
✔️ Avaliamos com métricas completas  
✔️ Geramos curva ROC e AUC  
✔️ Interpretamos importâncias  
✔️ Comparamos com Gradient Boosting e Random Forest  

Agora estamos prontos para:

**PARTE 6 — LightGBM: Intuição e Implementação.**

<a id="lgbm"></a>
# 6. LightGBM — Intuição e Implementação

LightGBM significa:

# **Light Gradient Boosting Machine**

Ele foi criado pela Microsoft e é famoso por:

✔️ ser extremamente rápido  
✔️ treinar em milhões de linhas  
✔️ usar pouca memória  
✔️ ter performance comparável (ou superior) ao XGBoost  
✔️ lidar muito bem com variáveis categóricas  

É um dos modelos mais usados em produção.

---
# 6.1 Por que LightGBM é tão rápido?

LightGBM introduz duas ideias revolucionárias:

## ✔️ 1. Geração de histogramas
Em vez de usar valores contínuos, LightGBM:
- agrupa valores em bins  
- reduz custo computacional  
- acelera splits  

## ✔️ 2. Crescimento leaf‑wise (ao invés de level‑wise)

XGBoost cresce assim:
```
nível 0
├── nível 1
│   ├── nível 2
│   └── nível 2
```

LightGBM cresce assim:
```
folha mais promissora
└── divide
    └── divide
        └── divide
```

Isso gera:
- árvores mais profundas  
- melhor ajuste  
- maior risco de overfitting (controlável)  

Mas com **muito mais velocidade**.

---
# 6.2 Instalando e importando LightGBM

Caso não esteja instalado:

```
pip install lightgbm
```

In [ ]:
import lightgbm as lgb

---
# 6.3 Preparando os dados

In [ ]:
X = ratings_df[["beauty", "age", "students", "female_dummy"]]
y = ratings_df["high_eval"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

---
# 6.4 Ajustando o LightGBM Classifier

In [ ]:
lgb_model = lgb.LGBMClassifier(
    n_estimators=400,
    learning_rate=0.05,
    max_depth=-1,          # -1 = sem limite
    num_leaves=31,         # parâmetro mais importante do LightGBM
    subsample=0.9,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

lgb_model.fit(X_train, y_train)

---
# 6.5 Avaliando o modelo

In [ ]:
y_pred_lgb = lgb_model.predict(X_test)

results_lgb = pd.DataFrame({
    "Acurácia": [accuracy_score(y_test, y_pred_lgb)],
    "Precisão": [precision_score(y_test, y_pred_lgb)],
    "Recall": [recall_score(y_test, y_pred_lgb)],
    "F1": [f1_score(y_test, y_pred_lgb)]
})

results_lgb

🟣 **Interpretação**

LightGBM normalmente:
- é tão bom quanto XGBoost  
- ou até melhor em datasets grandes  
- com velocidade muito superior  

Aqui ele já mostra excelente desempenho.

---
# 6.6 Matriz de Confusão

In [ ]:
cm = confusion_matrix(y_test, y_pred_lgb)

plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Greens")
plt.xlabel("Previsto")
plt.ylabel("Real")
plt.title("Matriz de Confusão — LightGBM")
plt.show()

---
# 6.7 Curva ROC e AUC

In [ ]:
y_proba_lgb = lgb_model.predict_proba(X_test)[:, 1]

fpr, tpr, thresholds = roc_curve(y_test, y_proba_lgb)
auc_value = auc(fpr, tpr)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f"LightGBM (AUC = {auc_value:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
plt.xlabel("FPR")
plt.ylabel("TPR")
plt.title("Curva ROC — LightGBM")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

auc_value

---
# 6.8 Importância das Variáveis

In [ ]:
importance_df = pd.DataFrame({
    "Variável": X.columns,
    "Importância": lgb_model.feature_importances_
}).sort_values("Importância", ascending=False)

importance_df

In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(
    data=importance_df,
    x="Importância",
    y="Variável",
    palette="viridis"
)
plt.title("Importância das Variáveis — LightGBM")
plt.xlabel("Importância")
plt.ylabel("Variável")
plt.show()

---
# 6.9 Comparando LightGBM vs XGBoost vs Gradient Boosting vs Random Forest

In [ ]:
comparison = pd.DataFrame({
    "Modelo": ["Random Forest", "Gradient Boosting", "XGBoost", "LightGBM"],
    "Acurácia": [
        accuracy_score(y_test, y_pred_rf),
        accuracy_score(y_test, y_pred_gb),
        accuracy_score(y_test, y_pred_xgb),
        accuracy_score(y_test, y_pred_lgb)
    ],
    "Precisão": [
        precision_score(y_test, y_pred_rf),
        precision_score(y_test, y_pred_gb),
        precision_score(y_test, y_pred_xgb),
        precision_score(y_test, y_pred_lgb)
    ],
    "Recall": [
        recall_score(y_test, y_pred_rf),
        recall_score(y_test, y_pred_gb),
        recall_score(y_test, y_pred_xgb),
        recall_score(y_test, y_pred_lgb)
    ],
    "F1": [
        f1_score(y_test, y_pred_rf),
        f1_score(y_test, y_pred_gb),
        f1_score(y_test, y_pred_xgb),
        f1_score(y_test, y_pred_lgb)
    ]
})

comparison.set_index("Modelo")

🟣 **Interpretação**

- LightGBM e XGBoost geralmente lideram  
- Gradient Boosting clássico fica logo atrás  
- Random Forest é o mais robusto, mas menos preciso  

Em datasets grandes, LightGBM costuma ser o mais rápido.

---
# 6.10 Conclusão desta parte

✔️ Entendemos a intuição do LightGBM  
✔️ Ajustamos um modelo real  
✔️ Avaliamos com métricas completas  
✔️ Geramos curva ROC e AUC  
✔️ Interpretamos importâncias  
✔️ Comparamos com XGBoost e Gradient Boosting  

Agora estamos prontos para:

**PARTE 7 — CatBoost: Intuição e Implementação.**

<a id="cat"></a>
# 7. CatBoost — Intuição e Implementação

CatBoost significa:

# **Categorical Boosting**

Ele foi criado para resolver dois problemas clássicos:

## ✔️ 1. Como lidar com variáveis categóricas sem one‑hot?
CatBoost transforma categorias usando:
- estatísticas ordenadas  
- codificação baseada em permutações  

Isso evita:
- explosão de dimensionalidade  
- overfitting  
- perda de informação  

## ✔️ 2. Como evitar overfitting em boosting?
CatBoost usa:
- *Ordered Boosting*  
- *Oblivious Trees* (árvores simétricas)  

Isso gera:
- estabilidade  
- velocidade  
- excelente generalização  

CatBoost é um dos modelos mais fáceis de usar e mais robustos.

---
# 7.1 Instalando e importando CatBoost

Caso não esteja instalado:

```
pip install catboost
```

In [ ]:
from catboost import CatBoostClassifier

---
# 7.2 Preparando os dados

In [ ]:
X = ratings_df[["beauty", "age", "students", "female_dummy"]]
y = ratings_df["high_eval"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

---
# 7.3 Ajustando o CatBoost Classifier

CatBoost tem um grande diferencial:
- **não precisa de normalização**
- **não precisa de one‑hot**
- **não precisa de pré‑processamento**

Ele funciona direto nos dados.

In [ ]:
cat_model = CatBoostClassifier(
    iterations=400,
    learning_rate=0.05,
    depth=4,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=False
)

cat_model.fit(X_train, y_train)

---
# 7.4 Avaliando o modelo

In [ ]:
y_pred_cat = cat_model.predict(X_test)

results_cat = pd.DataFrame({
    "Acurácia": [accuracy_score(y_test, y_pred_cat)],
    "Precisão": [precision_score(y_test, y_pred_cat)],
    "Recall": [recall_score(y_test, y_pred_cat)],
    "F1": [f1_score(y_test, y_pred_cat)]
})

results_cat

🟣 **Interpretação**

CatBoost normalmente:
- tem recall muito forte  
- é extremamente estável  
- supera XGBoost e LightGBM quando há categorias  

Mesmo sem categorias aqui, ele já mostra excelente desempenho.

---
# 7.5 Matriz de Confusão

In [ ]:
cm = confusion_matrix(y_test, y_pred_cat)

plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Oranges")
plt.xlabel("Previsto")
plt.ylabel("Real")
plt.title("Matriz de Confusão — CatBoost")
plt.show()

---
# 7.6 Curva ROC e AUC

In [ ]:
y_proba_cat = cat_model.predict_proba(X_test)[:, 1]

fpr, tpr, thresholds = roc_curve(y_test, y_proba_cat)
auc_value = auc(fpr, tpr)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f"CatBoost (AUC = {auc_value:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
plt.xlabel("FPR")
plt.ylabel("TPR")
plt.title("Curva ROC — CatBoost")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

auc_value

---
# 7.7 Importância das Variáveis

In [ ]:
importance_df = pd.DataFrame({
    "Variável": X.columns,
    "Importância": cat_model.get_feature_importance()
}).sort_values("Importância", ascending=False)

importance_df

In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(
    data=importance_df,
    x="Importância",
    y="Variável",
    palette="inferno"
)
plt.title("Importância das Variáveis — CatBoost")
plt.xlabel("Importância")
plt.ylabel("Variável")
plt.show()

---
# 7.8 Comparando CatBoost vs LightGBM vs XGBoost vs Gradient Boosting vs Random Forest

In [ ]:
comparison = pd.DataFrame({
    "Modelo": ["Random Forest", "Gradient Boosting", "XGBoost", "LightGBM", "CatBoost"],
    "Acurácia": [
        accuracy_score(y_test, y_pred_rf),
        accuracy_score(y_test, y_pred_gb),
        accuracy_score(y_test, y_pred_xgb),
        accuracy_score(y_test, y_pred_lgb),
        accuracy_score(y_test, y_pred_cat)
    ],
    "Precisão": [
        precision_score(y_test, y_pred_rf),
        precision_score(y_test, y_pred_gb),
        precision_score(y_test, y_pred_xgb),
        precision_score(y_test, y_pred_lgb),
        precision_score(y_test, y_pred_cat)
    ],
    "Recall": [
        recall_score(y_test, y_pred_rf),
        recall_score(y_test, y_pred_gb),
        recall_score(y_test, y_pred_xgb),
        recall_score(y_test, y_pred_lgb),
        recall_score(y_test, y_pred_cat)
    ],
    "F1": [
        f1_score(y_test, y_pred_rf),
        f1_score(y_test, y_pred_gb),
        f1_score(y_test, y_pred_xgb),
        f1_score(y_test, y_pred_lgb),
        f1_score(y_test, y_pred_cat)
    ]
})

comparison.set_index("Modelo")

🟣 **Interpretação**

- CatBoost é extremamente competitivo  
- Em muitos cenários, ele vence LightGBM e XGBoost  
- É o modelo mais fácil de usar entre os três  

Em datasets com muitas categorias, CatBoost costuma ser o campeão.

---
# 7.9 Conclusão desta parte

✔️ Entendemos a intuição do CatBoost  
✔️ Ajustamos um modelo real  
✔️ Avaliamos com métricas completas  
✔️ Geramos curva ROC e AUC  
✔️ Interpretamos importâncias  
✔️ Comparamos com LightGBM e XGBoost  

Agora estamos prontos para:

**PARTE 8 — Importância das Variáveis no Boosting (comparação entre modelos).**

<a id="importancia"></a>
# 8. Importância das Variáveis no Boosting

Agora que treinamos:
- Gradient Boosting  
- XGBoost  
- LightGBM  
- CatBoost  

Vamos comparar como cada modelo calcula e interpreta **importância das variáveis**.

Isso é fundamental porque:
- cada algoritmo usa um critério diferente  
- importâncias não são diretamente comparáveis  
- mas padrões gerais podem ser observados  

---
# 8.1 Importância no Gradient Boosting (sklearn)

Baseada em:
- redução de impureza (Gini ou entropia)  
- soma dos ganhos em cada split  

In [ ]:
gb_importance = pd.DataFrame({
    "Variável": X.columns,
    "GB Importance": gb_final.feature_importances_
})

gb_importance

---
# 8.2 Importância no XGBoost

XGBoost usa vários tipos de importância:
- **gain** (padrão) → quanto cada variável melhora a perda  
- cover → número de amostras afetadas  
- weight → frequência de uso nos splits  

Aqui usamos **gain**.

In [ ]:
xgb_importance = pd.DataFrame({
    "Variável": X.columns,
    "XGB Importance": xgb_model.feature_importances_
})

xgb_importance

---
# 8.3 Importância no LightGBM

LightGBM usa:
- **split importance** (frequência de uso)  
- **gain importance** (ganho acumulado)  

Aqui usamos a importância padrão (split importance).

In [ ]:
lgb_importance = pd.DataFrame({
    "Variável": X.columns,
    "LGB Importance": lgb_model.feature_importances_
})

lgb_importance

---
# 8.4 Importância no CatBoost

CatBoost usa:
- **PredictionValuesChange**  
que mede quanto cada variável altera a previsão média.

É uma das importâncias mais estáveis.

In [ ]:
cat_importance = pd.DataFrame({
    "Variável": X.columns,
    "CAT Importance": cat_model.get_feature_importance()
})

cat_importance

---
# 8.5 Unificando todas as importâncias

In [ ]:
importance_all = (
    gb_importance
    .merge(xgb_importance, on="Variável")
    .merge(lgb_importance, on="Variável")
    .merge(cat_importance, on="Variável")
)

importance_all

---
# 8.6 Visualização comparativa

In [ ]:
importance_all_melted = importance_all.melt(
    id_vars="Variável",
    var_name="Modelo",
    value_name="Importância"
)

plt.figure(figsize=(10, 6))
sns.barplot(
    data=importance_all_melted,
    x="Importância",
    y="Variável",
    hue="Modelo",
    palette="tab10"
)
plt.title("Comparação de Importância das Variáveis entre Modelos de Boosting")
plt.xlabel("Importância")
plt.ylabel("Variável")
plt.legend(title="Modelo")
plt.show()

🟣 **Interpretação**

Mesmo com métodos diferentes, padrões consistentes aparecem:

- **beauty** é a variável mais importante em todos os modelos  
- **students** e **age** aparecem como moderadamente importantes  
- **female_dummy** tem importância baixa  

Isso mostra que:
- boosting é robusto  
- diferentes algoritmos convergem para padrões semelhantes  

---
# 8.7 Normalizando importâncias para comparação justa

In [ ]:
importance_norm = importance_all.copy()
for col in importance_norm.columns[1:]:
    importance_norm[col] = importance_norm[col] / importance_norm[col].max()

importance_norm

---
# 8.8 Visualização das importâncias normalizadas

In [ ]:
importance_norm_melted = importance_norm.melt(
    id_vars="Variável",
    var_name="Modelo",
    value_name="Importância Normalizada"
)

plt.figure(figsize=(10, 6))
sns.barplot(
    data=importance_norm_melted,
    x="Importância Normalizada",
    y="Variável",
    hue="Modelo",
    palette="tab20"
)
plt.title("Importância Normalizada — Comparação entre Modelos de Boosting")
plt.xlabel("Importância Normalizada")
plt.ylabel("Variável")
plt.legend(title="Modelo")
plt.show()

---
# 8.9 Conclusões importantes sobre importâncias

✔️ Importância ≠ causalidade  
✔️ Importância ≠ efeito marginal  
✔️ Importância depende do algoritmo  
✔️ Importância depende da estrutura das árvores  
✔️ Importância é relativa, não absoluta  

Mas:
- padrões consistentes entre modelos são confiáveis  
- boosting é robusto na identificação de variáveis relevantes  

E isso é extremamente útil em:
- explicabilidade  
- auditoria  
- seleção de variáveis  
- relatórios executivos  

---
# 8.10 Conclusão desta parte

✔️ Comparamos importâncias entre Gradient Boosting, XGBoost, LightGBM e CatBoost  
✔️ Visualizamos padrões consistentes  
✔️ Normalizamos importâncias para comparação justa  
✔️ Interpretamos diferenças entre algoritmos  
✔️ Consolidamos conhecimento de explicabilidade em boosting  

Agora estamos prontos para:

**PARTE 9 — Projeto Final: Boosting Completo (XGBoost + LightGBM + CatBoost).**

<a id="projeto"></a>
# 9. Projeto Final — Boosting Completo

Neste projeto, vamos:

✔️ Treinar Gradient Boosting  
✔️ Treinar XGBoost  
✔️ Treinar LightGBM  
✔️ Treinar CatBoost  
✔️ Avaliar todos com métricas completas  
✔️ Comparar desempenho  
✔️ Gerar curvas ROC e AUC  
✔️ Comparar importâncias das variáveis  

Este é o pipeline final do Módulo 10.

---
# 9.1 Preparação dos dados

In [ ]:
X = ratings_df[["beauty", "age", "students", "female_dummy"]]
y = ratings_df["high_eval"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

---
# 9.2 Treinando todos os modelos

In [ ]:
# Gradient Boosting
gb = GradientBoostingClassifier(
    n_estimators=400,
    learning_rate=0.05,
    max_depth=3,
    subsample=0.9,
    random_state=42
)
gb.fit(X_train, y_train)

# XGBoost
xgb_model = xgb.XGBClassifier(
    n_estimators=400,
    learning_rate=0.05,
    max_depth=3,
    subsample=0.9,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)
xgb_model.fit(X_train, y_train)

# LightGBM
lgb_model = lgb.LGBMClassifier(
    n_estimators=400,
    learning_rate=0.05,
    num_leaves=31,
    subsample=0.9,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)
lgb_model.fit(X_train, y_train)

# CatBoost
cat_model = CatBoostClassifier(
    iterations=400,
    learning_rate=0.05,
    depth=4,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=False
)
cat_model.fit(X_train, y_train)

---
# 9.3 Previsões

In [ ]:
y_pred_gb = gb.predict(X_test)
y_pred_xgb = xgb_model.predict(X_test)
y_pred_lgb = lgb_model.predict(X_test)
y_pred_cat = cat_model.predict(X_test)

---
# 9.4 Avaliação completa

In [ ]:
def metrics(y_true, y_pred):
    return [
        accuracy_score(y_true, y_pred),
        precision_score(y_true, y_pred),
        recall_score(y_true, y_pred),
        f1_score(y_true, y_pred)
    ]

results = pd.DataFrame({
    "Modelo": ["Gradient Boosting", "XGBoost", "LightGBM", "CatBoost"],
    "Acurácia": [
        metrics(y_test, y_pred_gb)[0],
        metrics(y_test, y_pred_xgb)[0],
        metrics(y_test, y_pred_lgb)[0],
        metrics(y_test, y_pred_cat)[0]
    ],
    "Precisão": [
        metrics(y_test, y_pred_gb)[1],
        metrics(y_test, y_pred_xgb)[1],
        metrics(y_test, y_pred_lgb)[1],
        metrics(y_test, y_pred_cat)[1]
    ],
    "Recall": [
        metrics(y_test, y_pred_gb)[2],
        metrics(y_test, y_pred_xgb)[2],
        metrics(y_test, y_pred_lgb)[2],
        metrics(y_test, y_pred_cat)[2]
    ],
    "F1": [
        metrics(y_test, y_pred_gb)[3],
        metrics(y_test, y_pred_xgb)[3],
        metrics(y_test, y_pred_lgb)[3],
        metrics(y_test, y_pred_cat)[3]
    ]
})

results.set_index("Modelo")

🟣 **Interpretação**

- CatBoost e XGBoost geralmente lideram  
- LightGBM é extremamente competitivo  
- Gradient Boosting clássico fica logo atrás  

Em dados tabulares, boosting costuma superar Random Forest.

---
# 9.5 Curvas ROC e AUC

In [ ]:
models = {
    "Gradient Boosting": gb,
    "XGBoost": xgb_model,
    "LightGBM": lgb_model,
    "CatBoost": cat_model
}

plt.figure(figsize=(8, 6))

for name, model in models.items():
    y_proba = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc_value = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc_value:.3f})")

plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
plt.xlabel("FPR")
plt.ylabel("TPR")
plt.title("Curvas ROC — Comparação entre Modelos de Boosting")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

---
# 9.6 Importância das Variáveis — Comparação

In [ ]:
importance_df = pd.DataFrame({
    "Variável": X.columns,
    "GB": gb.feature_importances_,
    "XGB": xgb_model.feature_importances_,
    "LGB": lgb_model.feature_importances_,
    "CAT": cat_model.get_feature_importance()
})

importance_df

In [ ]:
importance_melted = importance_df.melt(
    id_vars="Variável",
    var_name="Modelo",
    value_name="Importância"
)

plt.figure(figsize=(10, 6))
sns.barplot(
    data=importance_melted,
    x="Importância",
    y="Variável",
    hue="Modelo",
    palette="tab10"
)
plt.title("Importância das Variáveis — Comparação entre Modelos de Boosting")
plt.xlabel("Importância")
plt.ylabel("Variável")
plt.legend(title="Modelo")
plt.show()

---
# 9.7 Conclusões do Projeto Final

✔️ Treinamos quatro modelos de boosting  
✔️ Avaliamos com métricas completas  
✔️ Geramos curvas ROC e AUC  
✔️ Comparamos importâncias das variáveis  
✔️ Construímos um pipeline profissional  

**Boosting é o estado da arte para dados tabulares.**

- XGBoost → excelente equilíbrio entre velocidade e precisão  
- LightGBM → mais rápido, ótimo para datasets grandes  
- CatBoost → melhor para variáveis categóricas  
- Gradient Boosting → baseline sólido  

Você agora domina todo o ecossistema de boosting moderno.

---
# 🎉 Fim do Módulo 10

Parabéns!  
Você concluiu um dos módulos mais importantes do curso.

O próximo passo é avançar para:

# **Módulo 11 — Redes Neurais e Deep Learning (Introdução Prática).**